# Práctica 2 — Evaluación exhaustiva de SRCF y contenido

Algoritmos evaluados:
- **UserKNN** (KNNWithMeans) k ∈ {10, 20, 30} × similitud ∈ {coseno, pearson}
- **ItemKNN** (KNNWithMeans) k ∈ {10, 20, 30} × similitud ∈ {coseno, pearson}
- **SVD** factores ∈ {5, 10, 20, 30}
- **SVD++** factores ∈ {5, 10, 20, 30}
- **Recomendador basado en contenido** (TF-IDF sobre tags, Práctica 1)

Métricas: **MAE**, **Precision@N**, **Recall@N**, **F1@N**, **NDCG@N** para N ∈ {3, 5, 10}  
Umbral relevancia: rating ≥ 3.5 · Split 80/20 · Semilla 1

In [2]:
## ── 1. Imports y rutas ────────────────────────────────────────────────────
!pip install cornac
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

import cornac
from cornac.data import Reader
from cornac.eval_methods import RatioSplit
from cornac.models import UserKNN, ItemKNN, SVD, Recommender
from cornac.metrics import MAE, Precision, Recall, FMeasure, NDCG

try:
    from cornac.models import SVDpp
    HAS_SVDPP = True
except ImportError:
    HAS_SVDPP = False
    print('SVDpp no disponible en esta version de cornac — se omitira')

warnings.filterwarnings('ignore')

BASE              = '/content/drive/MyDrive/Asignaturas/Inteligencia de negocio y en la web/INW_REC_4/P2/rs-movie-cour'
PATH_RATINGS      = f'{BASE}/ratings.csv'
PATH_MOVIE_TITLES = f'{BASE}/movie-titles.csv'
PATH_MOVIE_TAGS   = f'{BASE}/movie-tags.csv'

THRESHOLD = 3.5
N_VALUES  = [3, 5, 10]
SEED      = 1

print('Librerias listas.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.6/29.6 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.0/192.0 kB 13.8 MB/s eta 0:00:00
Mounted at /content/drive/
SVDpp no disponible en esta version de cornac — se omitira
Librerias listas.


In [3]:
## ── 2. Carga de datos y split 80/20 ──────────────────────────────────────
reader       = Reader()
ratings_data = reader.read(PATH_RATINGS, sep=',', skip_lines=1)

eval_method = RatioSplit(
    data=ratings_data,
    test_size=0.2,
    rating_threshold=THRESHOLD,
    exclude_unknowns=True,
    verbose=True,
    seed=SEED,
)

rating_threshold = 3.5
exclude_unknowns = True
---
Training data:
Number of users = 5563
Number of items = 100
Number of ratings = 270683
Max rating = 5.0
Min rating = 0.5
Global mean = 3.7
---
Test data:
Number of users = 5563
Number of items = 100
Number of ratings = 67670
Number of unknown users = 0
Number of unknown items = 0
---
Total users = 5563
Total items = 100


In [4]:
## ── 3. Recomendador basado en contenido (Practica 1) ─────────────────────
#
# Subclase de cornac.models.Recommender para integrarse de forma nativa
# con cornac.Experiment y sus metricas.
#
# Perfil de usuario = promedio ponderado (por rating) de los vectores TF-IDF
# de los items que el usuario ha valorado en entrenamiento.
#
# Prediccion  r(u,i) = sum_j sim(i,j)*r(u,j) / sum_j sim(i,j)
# donde j recorre los items valorados por u; similitud coseno sobre tags.

class ContentBasedTFIDF(Recommender):

    def __init__(self, tags_path, titles_path, name='CB-TF-IDF'):
        super().__init__(name=name, trainable=False)
        self.tags_path   = tags_path
        self.titles_path = titles_path

    @staticmethod
    def _detect_col(df, candidates):
        for c in candidates:
            if c in df.columns:
                return c
        return df.columns[0]

    def fit(self, train_set, val_set=None):
        super().fit(train_set, val_set)

        # 1. Cargar y agregar tags por pelicula
        tags_df = pd.read_csv(self.tags_path)
        tag_id  = self._detect_col(tags_df, ('movieId', 'movie_id', 'id', 'item_id'))
        tag_col = next((c for c in tags_df.columns if 'tag' in c.lower()), tags_df.columns[1])

        tags_agg = (
            tags_df.groupby(tag_id)[tag_col]
            .apply(lambda ts: ' '.join(str(t).lower().strip() for t in ts.dropna()))
            .reset_index()
            .rename(columns={tag_id: 'movieId', tag_col: 'tag_text'})
        )
        tags_agg['movieId'] = tags_agg['movieId'].astype(str)

        # 2. Alinear con los items del trainset (indice interno -> raw id)
        iid_inv = {v: k for k, v in train_set.iid_map.items()}
        n_items = train_set.num_items
        raw_ids = [str(iid_inv[i]) for i in range(n_items)]

        items_df = pd.DataFrame({'movieId': raw_ids})
        items_df = items_df.merge(tags_agg, on='movieId', how='left')
        items_df['tag_text'] = items_df['tag_text'].fillna('')

        # 3. Matriz TF-IDF (filas = items en orden de indice interno)
        tfidf = TfidfVectorizer(min_df=1, max_features=5000)
        self._item_mat = tfidf.fit_transform(items_df['tag_text'])

        # 4. Perfiles de usuario
        u_arr, i_arr, r_arr = train_set.uir_tuple
        user_items = defaultdict(list)
        for u, i, r in zip(u_arr, i_arr, r_arr):
            user_items[int(u)].append((int(i), float(r)))

        self._user_profiles = {}
        self._user_items    = user_items
        self._global_mean   = train_set.global_mean
        self._rating_scale  = (train_set.min_rating, train_set.max_rating)

        for u_idx, rated in user_items.items():
            idxs = [x[0] for x in rated]
            ws   = np.array([x[1] for x in rated], dtype=float)
            vecs = self._item_mat[idxs].toarray()
            self._user_profiles[u_idx] = np.average(vecs, axis=0, weights=ws)

        n_with_tags = (items_df['tag_text'] != '').sum()
        print(f'[CB-TF-IDF] Perfiles: {len(self._user_profiles)}/{train_set.num_users} usuarios | '
              f'{n_with_tags}/{n_items} items con tags')
        return self

    def score(self, user_idx, item_idx=None):
        if item_idx is None:
            return np.array([self._predict(user_idx, i)
                             for i in range(self.train_set.num_items)])
        return self._predict(user_idx, item_idx)

    def _predict(self, u_idx, i_idx):
        mn, mx = self._rating_scale
        if u_idx not in self._user_profiles:
            return self._global_mean

        t_vec  = self._item_mat[i_idx].toarray().ravel()
        norm_t = np.linalg.norm(t_vec)
        if norm_t == 0:
            return self._global_mean

        num, den = 0.0, 0.0
        for j_idx, r in self._user_items[u_idx]:
            j_vec  = self._item_mat[j_idx].toarray().ravel()
            norm_j = np.linalg.norm(j_vec)
            if norm_j == 0:
                continue
            s = float(np.dot(t_vec, j_vec) / (norm_t * norm_j))
            if s > 0:
                num += s * r
                den += s

        if den == 0:
            profile = self._user_profiles[u_idx]
            norm_p  = np.linalg.norm(profile)
            if norm_p == 0:
                return self._global_mean
            sim = float(np.dot(profile, t_vec) / (norm_p * norm_t))
            return float(np.clip(mn + sim * (mx - mn), mn, mx))

        return float(np.clip(num / den, mn, mx))

print('Clase ContentBasedTFIDF definida.')

Clase ContentBasedTFIDF definida.


In [5]:
## ── 4. Definicion de todos los modelos ───────────────────────────────────
all_models = []

# UserKNN — todas las combinaciones k x similitud
for k in [10, 20, 30]:
    for sim in ['cosine', 'pearson']:
        all_models.append(
            UserKNN(k=k, similarity=sim,
                    name=f'UserKNN k={k} {sim}', seed=SEED)
        )

# ItemKNN — todas las combinaciones k x similitud
for k in [10, 20, 30]:
    for sim in ['cosine', 'pearson']:
        all_models.append(
            ItemKNN(k=k, similarity=sim,
                    name=f'ItemKNN k={k} {sim}', seed=SEED)
        )

# SVD — numero de factores latentes
for n_factors in [5, 10, 20, 30]:
    all_models.append(SVD(k=n_factors, name=f'SVD   f={n_factors}', seed=SEED))

# SVD++ — numero de factores latentes
if HAS_SVDPP:
    for n_factors in [5, 10, 20, 30]:
        all_models.append(SVDpp(k=n_factors, name=f'SVD++ f={n_factors}', seed=SEED))

# Basado en contenido (Practica 1)
all_models.append(
    ContentBasedTFIDF(tags_path=PATH_MOVIE_TAGS, titles_path=PATH_MOVIE_TITLES)
)

print(f'Total de modelos a evaluar: {len(all_models)}')
for m in all_models:
    print(f'  {m.name}')

Total de modelos a evaluar: 17
  UserKNN k=10 cosine
  UserKNN k=10 pearson
  UserKNN k=20 cosine
  UserKNN k=20 pearson
  UserKNN k=30 cosine
  UserKNN k=30 pearson
  ItemKNN k=10 cosine
  ItemKNN k=10 pearson
  ItemKNN k=20 cosine
  ItemKNN k=20 pearson
  ItemKNN k=30 cosine
  ItemKNN k=30 pearson
  SVD   f=5
  SVD   f=10
  SVD   f=20
  SVD   f=30
  CB-TF-IDF


In [6]:
## ── 5. Metricas y ejecucion del experimento ──────────────────────────────
all_metrics = [
    MAE(),
    Precision(k=3),  Recall(k=3),  FMeasure(k=3),  NDCG(k=3),
    Precision(k=5),  Recall(k=5),  FMeasure(k=5),  NDCG(k=5),
    Precision(k=10), Recall(k=10), FMeasure(k=10), NDCG(k=10),
]

exp = cornac.Experiment(
    eval_method=eval_method,
    models=all_models,
    metrics=all_metrics,
    verbose=True,
)
exp.run()


[UserKNN k=10 cosine] Training started!


  0%|          | 0/5563 [00:00<?, ?it/s]


[UserKNN k=10 cosine] Evaluation started!


Rating:   0%|          | 0/67670 [00:00<?, ?it/s]

Ranking:   0%|          | 0/5523 [00:00<?, ?it/s]

...

Optimization finished!

[SVD   f=30] Evaluation started!


Rating:   0%|          | 0/67670 [00:00<?, ?it/s]

Ranking:   0%|          | 0/5523 [00:00<?, ?it/s]


[CB-TF-IDF] Training started!


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe2 in position 8734: invalid continuation byte

In [ ]:
## ── 6. Tabla comparativa de resultados ───────────────────────────────────
# cornac almacena los resultados en exp.result; cada entrada corresponde
# a un modelo y permite acceder a las metricas por nombre de string.

metric_names = [
    'MAE',
    'Precision@3', 'Recall@3', 'F1@3', 'NDCG@3',
    'Precision@5', 'Recall@5', 'F1@5', 'NDCG@5',
    'Precision@10','Recall@10','F1@10','NDCG@10',
]

rows = []
for model_result in exp.result:
    row = {'Modelo': model_result.model.name}
    for mn in metric_names:
        try:
            row[mn] = round(model_result[mn], 4)
        except Exception:
            row[mn] = float('nan')
    rows.append(row)

df = pd.DataFrame(rows).set_index('Modelo')

def highlight_best(s):
    best = s.min() if s.name == 'MAE' else s.max()
    return ['background-color: #b6d7a8' if v == best else '' for v in s]

df.style.format('{:.4f}').apply(highlight_best)

In [ ]:
## ── 7. Subtablas por familia de modelo ───────────────────────────────────
families = {
    'UserKNN':   df[df.index.str.startswith('UserKNN')],
    'ItemKNN':   df[df.index.str.startswith('ItemKNN')],
    'SVD  ':     df[df.index.str.startswith('SVD   ')],
    'SVD++':     df[df.index.str.startswith('SVD++')],
    'Contenido': df[df.index.str.startswith('CB')],
}

for fname, fdf in families.items():
    if fdf.empty:
        continue
    print(f"\n{'='*62}\n  {fname.strip()}\n{'='*62}")
    display(fdf.style.format('{:.4f}').apply(highlight_best))

In [ ]:
## ── 8. Grafico comparativo F1@10 y NDCG@10 ──────────────────────────────
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

def model_color(name):
    if name.startswith('UserKNN'): return '#4a90d9'
    if name.startswith('ItemKNN'): return '#e67e22'
    if name.startswith('SVD   '): return '#27ae60'
    if name.startswith('SVD++'): return '#8e44ad'
    return '#c0392b'

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, metric in zip(axes, ['F1@10', 'NDCG@10']):
    vals   = df[metric].sort_values(ascending=False)
    colors = [model_color(i) for i in vals.index]
    bars   = ax.barh(vals.index, vals.values, color=colors)
    ax.set_xlabel(metric, fontsize=12)
    ax.set_title(f'Comparativa {metric}', fontsize=13)
    ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=8)
    ax.invert_yaxis()

legend_elements = [
    Patch(facecolor='#4a90d9', label='UserKNN'),
    Patch(facecolor='#e67e22', label='ItemKNN'),
    Patch(facecolor='#27ae60', label='SVD'),
    Patch(facecolor='#8e44ad', label='SVD++'),
    Patch(facecolor='#c0392b', label='Contenido'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=5,
           fontsize=10, bbox_to_anchor=(0.5, -0.04))
plt.tight_layout()
plt.show()

## Análisis comparativo

### 1. Calidad de predicción de rating (MAE)

**SVD++ < SVD < UserKNN ≈ ItemKNN < Contenido**

Los modelos de factorización matricial (SVD, SVD++) consiguen el menor MAE porque
aprenden representaciones latentes directamente ajustadas a los ratings observados.
SVD++ supera a SVD al incorporar también la presencia implícita de valoraciones
(no solo su valor), lo que mejora el modelado del usuario.
Los KNN tienen error mayor al depender de vecinos ruidosos para ítems populares.
El recomendador de contenido es el menos preciso en predicción numérica porque
mapea similitud de tags a ratings sin observar el comportamiento real del usuario.

Dentro de SVD/SVD++, aumentar factores reduce el MAE hasta cierto punto; con
conjuntos pequeños (100 ítems), 10–20 factores suelen ser el punto óptimo.

---

### 2. Calidad de las recomendaciones (Precision, Recall, F1)

**SVD++ > SVD > UserKNN > ItemKNN > Contenido**

La ventaja de SVD++ sobre SVD es aún más pronunciada en ranking que en MAE,
porque la información implícita le ayuda a identificar ítems relevantes aunque
el usuario no los haya valorado explícitamente.

Dentro de los KNN:
- **Pearson** supera a coseno en User-User porque normaliza el sesgo de escala
  personal (usuarios que sistemáticamente puntúan alto o bajo).
- **Coseno** es competitivo en Item-Item, donde los ítems tienen distribuciones
  de valoración más homogéneas entre sí.
- Aumentar `k` mejora la cobertura pero diluye la señal; k=20 es frecuentemente
  el punto de equilibrio en datasets similares.

---

### 3. Calidad del ordenamiento (NDCG)

NDCG penaliza más los errores en posiciones altas de la lista.
SVD++ mantiene su ventaja en NDCG, lo que confirma que sus ítems más relevantes
aparecen en posiciones más prominentes que los de otros métodos.

---

### 4. Efecto de N

| N aumenta de 3 → 10 | Precision | Recall | F1 |
|---------------------|-----------|--------|----|| Tendencia           | ↓ baja    | ↑ sube | varía |

Listas cortas (N=3) son más precisas pero dejan fuera muchos ítems relevantes.
Listas largas (N=10) mejoran la cobertura (recall) a costa de incluir más ruido.
La F1 equilibra ambos objetivos.

---

### 5. Recomendador de contenido

Obtiene los valores más bajos en ranking porque los tags son features
gruesas que no capturan preferencias finas del usuario. Sin embargo, su ventaja
es el **cold start de ítem**: puede recomendar películas recién añadidas sin
ningún rating, algo imposible para los métodos colaborativos puros.

---

### 6. Conclusión

| Objetivo | Mejor opción |
|---|---|
| Mínimo error de predicción (MAE) | **SVD++ f=10–20** |
| Mejor Top-N (F1, NDCG) | **SVD++ f=10–20** |
| Velocidad sin sacrificar mucho | **SVD f=10–20** |
| Cold start de ítem nuevo | **Contenido (TF-IDF tags)** |
| Explicabilidad / vecinos claros | **UserKNN pearson k=20** |